# ALCF IRI bash-skill benchmark (Claude Code driver)

**What this notebook does, top to bottom:**

1. **Preflight** — checks that `claude`, `curl`, `jq` are on PATH and that ALCF auth is set up.
2. **Configure** — pick which questions to run, how many trials, concurrency, and the judge model.
3. **Sweep** — spawns headless Claude Code (`claude -p --bare`) per question. Each subprocess is fresh — no shared memory across questions. Only `Bash` is enabled, and the ALCF IRI bash skill is appended to the system prompt.
4. **Judge** — scores each answer 0 or 1 with a strict binary rubric ("does the factual claim match the trace?" — style-blind).
5. **Report** — per-question table, overall accuracy, cost, wall time.

Just **Run All**. The full 16-question sweep takes 5–15 minutes and costs a few USD via the Anthropic API.

Prereqs:
- `claude` CLI (Claude Code 2.x) — used to run the coding-agent trials.
- `ANTHROPIC_API_KEY` — used both by Claude Code and by the binary judge.
- `ALCF_API_TOKEN`, or a valid cached Globus token at `~/.globus/app/8b84fc2d-.../alcf_facility_api_app/tokens.json` — the ALCF IRI API needs one of these. See `src/chemgraph/skills/alcf_iri_bash.md` § Auth for how to populate.
- `curl`, `jq` — the skill uses them.

## 1. Preflight

In [1]:
import os, shutil, subprocess, sys, json
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'src' / 'chemgraph').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'src' / 'chemgraph').exists(), (
    f'could not locate ChemGraph repo root from {Path.cwd()!s}. '
    'Open this notebook from within the checkout.'
)
print(f'repo root:      {REPO_ROOT}')

SKILL_PATH = REPO_ROOT / 'src' / 'chemgraph' / 'skills' / 'alcf_iri_bash.md'
assert SKILL_PATH.exists(), f'skill file missing at {SKILL_PATH}'
print(f'skill file:     {SKILL_PATH.relative_to(REPO_ROOT)}  ({SKILL_PATH.stat().st_size:,} bytes)')

# Binaries the skill needs
for bin_name in ('claude', 'curl', 'jq'):
    path = shutil.which(bin_name)
    assert path, f'{bin_name!r} not on PATH; install before running'
    print(f'{bin_name:<15} {path}')

# claude CLI version (surface it early so bug reports are easy)
ver = subprocess.run(['claude', '--version'], capture_output=True, text=True)
print(f'claude version: {ver.stdout.strip() or ver.stderr.strip()}')

# Claude Code auth check. The CLI accepts either $ANTHROPIC_API_KEY OR an
# OAuth session established via `claude login`. If neither is present, the
# sweep will fail with a clear Claude-Code error -- warn now instead of
# asserting so users who use OAuth aren't blocked here.
if os.environ.get('ANTHROPIC_API_KEY'):
    print('claude auth:    ANTHROPIC_API_KEY set in env')
else:
    print('claude auth:    ANTHROPIC_API_KEY not in env -- '
          'assuming `claude login` OAuth is active (harness will surface an error if not)')

# ALCF token check -- either env var, or on-disk Globus cache.
GLOBUS_CACHE = Path.home() / '.globus/app/8b84fc2d-49e9-49ea-b54d-b3a29a70cf31/alcf_facility_api_app/tokens.json'
if os.environ.get('ALCF_API_TOKEN'):
    print('ALCF_API_TOKEN: set in env')
elif GLOBUS_CACHE.exists():
    print(f'ALCF_API_TOKEN: not in env, but cache exists at {GLOBUS_CACHE}')
    # Try to pull an access token out and export it so Claude Code's subprocess sees it.
    try:
        data = json.loads(GLOBUS_CACHE.read_text())
        tok = (data.get('access_token')
               or data.get('data', {}).get('DEFAULT', {})
                       .get('6be511f6-a071-471f-9bc0-02a0d0836723', {})
                       .get('access_token'))
        if tok:
            os.environ['ALCF_API_TOKEN'] = tok
            print(f'                exported cached token to env ({len(tok)} chars)')
        else:
            print('                WARNING: cache exists but no access_token field found')
    except Exception as e:
        print(f'                WARNING: could not read cache: {e}')
else:
    raise SystemExit(
        'No ALCF_API_TOKEN and no on-disk cache. See '
        f'{SKILL_PATH.relative_to(REPO_ROOT)} § Auth for how to get one '
        '(easiest: `python ~/tools/alcf_facility_api_globus_token.py authenticate`).'
    )

# Make the harness importable
sys.path.insert(0, str(REPO_ROOT / 'examples' / 'iri'))
print('sys.path prepended with examples/iri/')

repo root:      /Users/jinchuli/projects/chemgraph-academy/ChemGraph
skill file:     src/chemgraph/skills/alcf_iri_bash.md  (14,407 bytes)
claude          /Users/jinchuli/.local/bin/claude
curl            /usr/bin/curl
jq              /usr/bin/jq
claude version: 2.1.154 (Claude Code)
claude auth:    ANTHROPIC_API_KEY not in env -- assuming `claude login` OAuth is active (harness will surface an error if not)
ALCF_API_TOKEN: not in env, but cache exists at /Users/jinchuli/.globus/app/8b84fc2d-49e9-49ea-b54d-b3a29a70cf31/alcf_facility_api_app/tokens.json
                exported cached token to env (91 chars)
sys.path prepended with examples/iri/


## 2. Configuration

In [ ]:
# Sweep parameters. Tweak these before Run All.

# Which questions to run. Default = all 16. Set to e.g. ['q1', 'q2'] for a quick smoke.
QIDS = None

# Runs per question. 1 for a fast pass; 3 if you want variance bars.
TRIALS = 1

# Parallel claude subprocesses. Claude Code isn't cheap; 2 is a safe default.
CONCURRENCY = 2

# --- Claude Code isolation --------------------------------------------------
# --bare passes Claude Code -> no ambient project state (memory, CLAUDE.md,
# hooks) influences the run. That's the fair signal on skill quality.
#
# GOTCHA: --bare disables Claude Code's OAuth keychain read, so it REQUIRES
# ANTHROPIC_API_KEY in env to authenticate. If you're only logged in via
# `claude login`, --bare will fail with "Not logged in · Please run /login".
# We auto-disable --bare when ANTHROPIC_API_KEY is absent so the sweep runs;
# the tradeoff is that ambient project state (memory, CLAUDE.md) can leak in.
BARE_REQUESTED = True
BARE = BARE_REQUESTED and bool(os.environ.get('ANTHROPIC_API_KEY'))
if BARE_REQUESTED and not BARE:
    print('NOTE: --bare requires ANTHROPIC_API_KEY (OAuth keychain is skipped '
          'under --bare). Falling back to non-bare so the sweep can use your '
          '`claude login` OAuth session. Set ANTHROPIC_API_KEY for a truly '
          'isolated skill-only measurement.')

# --- Judge model ------------------------------------------------------------
# Two backends supported:
#
#   'argo'      -- routes through the ALCF/Argo shim (same as iri_qeval.ipynb).
#                  No ANTHROPIC_API_KEY needed. Requires the argo-shim proxy
#                  running locally at ARGO_BASE_URL.
#   'anthropic' -- direct Anthropic API. Needs ANTHROPIC_API_KEY.
#
# Auto-detect: prefer argo if $ARGO_USER is set OR the argo-shim proxy
# responds, else fall back to anthropic if ANTHROPIC_API_KEY is set.
# Override manually by hard-coding JUDGE_PROVIDER below.

ARGO_BASE_URL = 'http://127.0.0.1:18085/argoapi/v1'
ARGO_USER     = os.environ.get('ARGO_USER', 'jinchu.li')

def _detect_judge_provider():
    # Explicit env override wins
    forced = os.environ.get('BENCH_JUDGE_PROVIDER')
    if forced in ('argo', 'anthropic'):
        return forced
    # Prefer argo if the shim looks reachable
    try:
        import urllib.request
        urllib.request.urlopen(ARGO_BASE_URL.rstrip('/v1') + '/', timeout=0.5)
        return 'argo'
    except Exception:
        pass
    if os.environ.get('ARGO_USER'):
        return 'argo'
    if os.environ.get('ANTHROPIC_API_KEY'):
        return 'anthropic'
    # Fallback: try argo anyway; the judge cell will surface a clean error.
    return 'argo'

JUDGE_PROVIDER = _detect_judge_provider()

# Model name matches the provider convention:
#   argo  -> 'argo:claude-opus-4.7' (see chemgraph.models.openai.ARGO_LOCAL_OPENAI_MODEL_MAP)
#   anthropic -> a shipped Anthropic model id
JUDGE_MODEL = {
    'argo':      'argo:claude-opus-4.7',
    'anthropic': 'claude-opus-4-20250514',
}[JUDGE_PROVIDER]

# Where to write the raw JSONL + scored JSON.
BENCH_JSONL = REPO_ROOT / 'examples' / 'iri' / 'bench_claude_code.jsonl'
SCORED_JSON = REPO_ROOT / 'examples' / 'iri' / 'bench_claude_code_scored.json'

print(f'qids:           {QIDS if QIDS else "all 16"}')
print(f'trials:         {TRIALS}')
print(f'concurrency:    {CONCURRENCY}')
print(f'bare requested: {BARE_REQUESTED}    (effective: {BARE})')
print(f'judge provider: {JUDGE_PROVIDER}')
print(f'judge model:    {JUDGE_MODEL}')
if JUDGE_PROVIDER == 'argo':
    print(f'  argo base:    {ARGO_BASE_URL}')
    print(f'  argo user:    {ARGO_USER}')

## 3. Run the sweep

Spawns one fresh `claude -p --bare` subprocess per (question, trial) with the ALCF IRI bash skill appended to its system prompt. Each subprocess only has `Bash` enabled; everything else (Read, Write, WebFetch, ...) is disabled so it must curl its way to the answer.

The cell shows a live progress line per completion. Full 16-question sweep at concurrency=2 takes ~5-15 min.

In [3]:
import asyncio
from tqdm.auto import tqdm
from bench_claude_code import run_all, QUESTIONS

pbar_total = (len(QIDS) if QIDS else len(QUESTIONS)) * TRIALS
pbar = tqdm(total=pbar_total, desc='claude-code sweep')

def _on_progress(done, total, row):
    tag = '✓' if row['ok'] else '✗'
    pbar.set_postfix_str(f'{tag} {row["qid"]} wall={row["wall_ms"]//1000}s cost=${row.get("cost_usd") or 0:.3f}')
    pbar.update(1)

rows = await run_all(
    qids=QIDS,
    trials=TRIALS,
    concurrency=CONCURRENCY,
    bare=BARE,
    out_path=BENCH_JSONL,
    on_progress=_on_progress,
)
pbar.close()

n_ok = sum(1 for r in rows if r['ok'])
total_cost = sum(r.get('cost_usd') or 0 for r in rows)
total_wall = sum(r['wall_ms'] for r in rows) / 1000
print()
print(f'{n_ok}/{len(rows)} runs completed without error')
print(f'total wall time: {total_wall:.1f}s  ({total_wall/len(rows):.1f}s per run)')
print(f'total API cost:  ${total_cost:.2f}  (${total_cost/len(rows):.3f} per run)')
print(f'raw output:      {BENCH_JSONL.relative_to(REPO_ROOT)}')


claude-code sweep:   0%|          | 0/16 [00:00<?, ?it/s]


0/16 runs completed without error
total wall time: 9.3s  (0.6s per run)
total API cost:  $0.00  ($0.000 per run)
raw output:      examples/iri/bench_claude_code.jsonl


## 4. Binary judge

Scores each answer 0 or 1 based ONLY on whether the final answer's factual claims match the trace. Ignores formatting, verbosity, hedging, tool-use style — the whole point of a binary judge is cross-runtime fairness. Same rubric can be used to compare Claude Code's answers against `single_agent_iri`'s answers without penalizing either for style.


In [4]:
from pydantic import BaseModel, Field
import json as _json
import asyncio

# ChemGraph's load_chat_model dispatches by model-name prefix:
#   'argo:...' -> argo shim (needs ARGO_USER + local proxy)
#   'claude-...' with anthropic-shipped id -> ChatAnthropic
# so both JUDGE_PROVIDER paths converge on the same call site.

# Argo shim's model list expects specific claude entries. If we're on argo,
# monkey-patch the map so it recognises the current opus family. This is
# lifted from iri_qeval.ipynb; keeping it inline avoids a cross-notebook
# import.
if JUDGE_PROVIDER == 'argo':
    import chemgraph.models.openai as _cg_openai
    _cg_openai.ARGO_LOCAL_OPENAI_MODEL_MAP.update({
        'argo:claude-opus-5':      'Claude Opus 5',
        'argo:claude-opus-4.8':    'Claude Opus 4.8',
        'argo:claude-opus-4.7':    'Claude Opus 4.7',
        'argo:claude-opus-4.6':    'Claude Opus 4.6',
        'argo:claude-sonnet-5':    'Claude Sonnet 5',
        'argo:claude-sonnet-4.6':  'Claude Sonnet 4.6',
        'argo:claude-haiku-4.5':   'Claude Haiku 4.5',
    })
    # Argo shim uses OpenAI-compatible client under the hood; a dummy key
    # keeps openai-python happy.
    os.environ.setdefault('OPENAI_API_KEY', 'dummy')

from chemgraph.models.loader import load_chat_model

if JUDGE_PROVIDER == 'argo':
    judge_llm = load_chat_model(
        model_name=JUDGE_MODEL,
        temperature=0.0,
        base_url=ARGO_BASE_URL,
        argo_user=ARGO_USER,
    )
else:
    judge_llm = load_chat_model(
        model_name=JUDGE_MODEL,
        temperature=0.0,
    )

JUDGE_BINARY_SYSTEM = '''You are a strict binary grader for AI agent answers to questions about the ALCF IRI Facility API.

You will see:
- The user's question.
- The trace of tool calls the agent made and the responses it got (may be truncated).
- The agent's final answer.

Return 1 if the final answer's factual claims are correct given the trace evidence, else 0.

RULES:
- Grade CONTENT, not style. Verbose is fine. Terse is fine. Headers, tables, prose -- ignore all of it. Only factual correctness matters.
- Scalar questions ("how many X?") require the correct scalar. If the answer says "10" and the trace supports 10, score 1. If the trace supports 10 but the answer says 12, score 0.
- Multi-part answers: score 1 only if ALL factual claims match the trace. Any material contradiction -> 0.
- "Correct given the trace" means: the claim is supported by, or reasonably inferable from, the tool responses shown. Truncated responses (`... (truncated)`) may hide the answer; if the visible portion is consistent with the claim, do not penalize.
- If the agent refused, deflected, or asked a clarifying question instead of answering, score 0.
- If the agent hallucinated a value not present in ANY tool response, score 0.
- Pagination gotcha: if the question asks for a count and the agent reported N based on ONE page of ~100 results, score 0 -- the true count is likely larger and the answer is unsupported.
- If the trace shows no tool calls at all, score 0.
'''

JUDGE_BINARY_USER = '''QUESTION:
{question}

TRACE (tool calls + results, oldest first):
{trace}

FINAL ANSWER:
{answer}

Reply with JSON only:
{{"score": 0 or 1, "rationale": "<one sentence citing the trace evidence>"}}'''


class JudgeBinaryVerdict(BaseModel):
    score: int = Field(..., ge=0, le=1)
    rationale: str = ''


async def judge_one(row: dict) -> dict:
    msg = JUDGE_BINARY_USER.format(
        question=row['question'],
        trace=row.get('trace_rendered') or '(no trace)',
        answer=row.get('answer') or '(no final answer)',
    )
    try:
        resp = await judge_llm.ainvoke([
            {'role': 'system', 'content': JUDGE_BINARY_SYSTEM},
            {'role': 'user', 'content': msg},
        ])
        raw = resp.content if hasattr(resp, 'content') else str(resp)
        # judge may wrap in ```json fences
        r = raw.strip()
        if r.startswith('```'):
            r = r.split('```')[1]
            if r.startswith('json'):
                r = r[4:]
        parsed = _json.loads(r.strip())
        v = JudgeBinaryVerdict(**parsed)
        return {'score': v.score, 'rationale': v.rationale, 'error': None}
    except Exception as e:
        return {'score': None, 'rationale': '', 'error': repr(e)}


# Score all rows (concurrent -- judge calls are I/O bound and cheap)
sem = asyncio.Semaphore(8)

async def _score(row):
    async with sem:
        v = await judge_one(row)
        return {**row, 'judge_binary': v}

scored = list(await asyncio.gather(*[_score(r) for r in rows]))
Path(SCORED_JSON).write_text(_json.dumps(scored, indent=2))
print(f'scored {len(scored)} runs -> {SCORED_JSON.relative_to(REPO_ROOT)}')

2026-08-25 11:28:45,531 - chemgraph.models.openai - INFO - Using custom base URL: http://127.0.0.1:18085/argoapi/v1
2026-08-25 11:28:45,531 - chemgraph.models.openai - INFO - Using OpenAI-style Argo model for local endpoint 'http://127.0.0.1:18085/argoapi/v1': 'argo:claude-opus-4.7' -> 'Claude Opus 4.7'
2026-08-25 11:28:45,531 - chemgraph.models.openai - INFO - Using Argo user from config/ARGO_USER/default: jinchu.li
2026-08-25 11:28:45,717 - chemgraph.models.openai - INFO - Requested model: Claude Opus 4.7
2026-08-25 11:28:45,717 - chemgraph.models.openai - INFO - OpenAI model loaded successfully


scored 16 runs -> examples/iri/bench_claude_code_scored.json


## 5. Report

In [5]:
from collections import defaultdict

# Per-question aggregation (avg across trials)
by_qid = defaultdict(list)
for r in scored:
    by_qid[r['qid']].append(r)

correct_total = sum(
    1 for r in scored
    if (r.get('judge_binary') or {}).get('score') == 1
)
graded_total = sum(
    1 for r in scored
    if (r.get('judge_binary') or {}).get('score') is not None
)

print(f'Claude Code + alcf_iri_bash.md   |  {correct_total}/{graded_total} correct  '
      f'({100*correct_total/graded_total:.1f}%)' if graded_total else 'no scored rows')
print()
print(f'{"qid":<5} {"pass":<6} {"turns":>6} {"tok":>7} {"wall_s":>8} {"cost":>7}  {"rationale":<80}')
print('-' * 120)
for qid, group in sorted(by_qid.items(), key=lambda kv: int(kv[0].lstrip('q'))):
    passes = [
        (r.get('judge_binary') or {}).get('score')
        for r in group
    ]
    passes_str = ''.join('✓' if p == 1 else ('✗' if p == 0 else '?') for p in passes)
    avg_turns = sum(r.get('turns') or 0 for r in group) / len(group)
    avg_tok = sum(r.get('total_tokens') or 0 for r in group) / len(group)
    avg_wall = sum(r['wall_ms'] for r in group) / len(group) / 1000
    avg_cost = sum(r.get('cost_usd') or 0 for r in group) / len(group)
    # one representative rationale per qid (first failed if any, else first)
    fails = [r for r in group if (r.get('judge_binary') or {}).get('score') == 0]
    rep = fails[0] if fails else group[0]
    rationale = (rep.get('judge_binary') or {}).get('rationale', '')[:80]
    print(f'{qid:<5} {passes_str:<6} {avg_turns:>6.1f} {avg_tok:>7.0f} {avg_wall:>8.1f} '
          f'${avg_cost:>6.3f}  {rationale}')


Claude Code + alcf_iri_bash.md   |  0/16 correct  (0.0%)

qid   pass    turns     tok   wall_s    cost  rationale                                                                       
------------------------------------------------------------------------------------------------------------------------
q1    ✗         0.0       0      0.9 $ 0.000  No trace and no final answer provided.
q2    ✗         0.0       0      0.8 $ 0.000  No trace and no final answer provided.
q3    ✗         0.0       0      0.5 $ 0.000  No tool calls were made and no final answer was provided.
q4    ✗         0.0       0      0.6 $ 0.000  No tool calls were made and no answer was provided.
q5    ✗         0.0       0      0.5 $ 0.000  No trace and no final answer provided.
q6    ✗         0.0       0      0.6 $ 0.000  No trace and no final answer provided.
q7    ✗         0.0       0      0.5 $ 0.000  No tool calls were made and no final answer was provided.
q8    ✗         0.0       0      0.5 $ 0.000  No

## 6. Drill-down on failures

Rerun after tweaking the skill to see which questions still fail and why. Prints the question, the agent's answer, and the last few tool calls.

In [6]:
fails = [
    r for r in scored
    if (r.get('judge_binary') or {}).get('score') == 0
]
if not fails:
    print('all runs passed under the binary judge')
else:
    print(f'{len(fails)} failing runs (of {len(scored)}):')
    print()
    for i, r in enumerate(fails, 1):
        print('=' * 80)
        print(f'[{i}] {r["qid"]} trial={r["trial"]}  wall={r["wall_ms"]//1000}s '
              f'turns={r.get("turns")} cost=${r.get("cost_usd") or 0:.3f}')
        print(f'    question: {r["question"]}')
        v = r.get('judge_binary') or {}
        print(f'    judge:    {v.get("rationale", "")}')
        print(f'    answer:   {(r.get("answer") or "").strip()[:300]}')
        # last 3 tool call lines from the trace
        trace_lines = (r.get('trace_rendered') or '').splitlines()
        tail = trace_lines[-6:] if len(trace_lines) > 6 else trace_lines
        print(f'    trace (last {len(tail)} lines):')
        for line in tail:
            print(f'      {line[:180]}')
        print()


16 failing runs (of 16):

[1] q1 trial=0  wall=0s turns=0 cost=$0.000
    question: How many total resources does ALCF facility API list?
    judge:    No trace and no final answer provided.
    answer:   
    trace (last 0 lines):

[2] q2 trial=0  wall=0s turns=0 cost=$0.000
    question: How many compute resources are currently marked "up"?
    judge:    No trace and no final answer provided.
    answer:   
    trace (last 0 lines):

[3] q3 trial=0  wall=0s turns=0 cost=$0.000
    question: What is the UUID of the resource named "Aurora"?
    judge:    No tool calls were made and no final answer was provided.
    answer:   
    trace (last 0 lines):

[4] q4 trial=0  wall=0s turns=0 cost=$0.000
    question: How many storage resources are listed (any status)?
    judge:    No tool calls were made and no answer was provided.
    answer:   
    trace (last 0 lines):

[5] q5 trial=0  wall=0s turns=0 cost=$0.000
    question: How many capabilities does the facility expose?
    judge:    N